# Credit Risk Scoring - Exploratory Data Analysis

Exploring the German Credit Dataset for the Credit Risk Scoring Service.

**Dataset:** Statlog (German Credit Data) - UCI Machine Learning Repository
**Rows:** 1,000  
**Features:** 20 (7 numeric, 13 categorical)  
**Target:** 0 = No default (Good), 1 = Default (Bad)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, map_target

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded successfully')

In [ ]:
# Load data
df = load_raw_data()
df = map_target(df)
print(f'Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Label distribution
target_counts = df['target'].value_counts()
print('Target Distribution:')
print(f'  0 (No Default): {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)')
print(f'  1 (Default):    {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#2ecc71', '#e74c3c']
ax[0].bar(['No Default (0)', 'Default (1)'], target_counts.values, color=colors)
ax[0].set_title('Target Class Distribution')
ax[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    ax[0].text(i, v + 10, str(v), ha='center')

ax[1].pie(target_counts.values, labels=['No Default (0)', 'Default (1)'],
          autopct='%1.1f%%', colors=colors, startangle=90)
ax[1].set_title('Target Class Proportions')
plt.tight_layout()
plt.show()

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print('Missing Value Analysis:')
print(missing_df[missing_df['Missing'] > 0] if missing_df['Missing'].sum() > 0 else 'No missing values found!')

# Data types
print('\nData Types:')
print(df.dtypes)

In [ ]:
# Numeric features distribution
numeric_cols = ['duration', 'credit_amount', 'installment_rate', 
                'residence_since', 'age', 'num_credits', 'people_maintenance']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:7]):
    # Histogram by target
    for target_val, color, label in [(0, '#2ecc71', 'No Default'), (1, '#e74c3c', 'Default')]:
        data = df[df['target'] == target_val][col]
        axes[i].hist(data, alpha=0.6, bins=20, color=color, label=label)
    axes[i].set_title(f'{col} by Target')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend()

# Hide unused subplots
for j in range(len(numeric_cols), 9):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics by target
print('Summary Statistics by Target:')
numeric_cols_full = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_full.remove('target')
summary = df.groupby('target')[numeric_cols_full].describe().T
summary

In [ ]:
# Categorical feature cardinality
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Categorical Feature Cardinality:')
for col in cat_cols:
    n_unique = df[col].nunique()
    print(f'  {col}: {n_unique} unique values')

# Default rate by top categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
top_cats = ['checking_status', 'credit_history', 'purpose', 'savings']
for i, col in enumerate(top_cats):
    cross = pd.crosstab(df[col], df['target'], normalize='index') * 100
    cross[1].sort_values(ascending=False).head(10).plot(
        kind='bar', ax=axes[i], color=['#e74c3c' if v > 30 else '#2ecc71' for v in cross[1].sort_values(ascending=False).head(10)]
    )
    axes[i].set_title(f'Default Rate by {col}')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation (numeric)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Key insights
print('=' * 60)
print('KEY INSIGHTS')
print('=' * 60)

print('\n1. Class Imbalance:')
print(f'   Dataset has {target_counts[0]} good (70%) and {target_counts[1]} bad (30%) credits.')
print('   Moderate imbalance - class weights recommended for training.')

print('\n2. No Missing Values:')
print('   The dataset has no missing values, simplifying preprocessing.')

print('\n3. Key Predictive Features (from literature):')
print('   - Checking account status (strongest predictor)')
print('   - Credit history')
print('   - Credit amount')
print('   - Duration')
print('   - Savings account')

print('\n4. Data Characteristics:')
print(f'   - {len(numeric_cols)} numeric features with varying scales (need scaling)')
print(f'   - {len(cat_cols)} categorical features with {sum(df[c].nunique() for c in cat_cols)} total categories (need encoding)')

print('\n5. Preprocessing Requirements:')
print('   - Feature scaling for numeric columns')
print('   - One-hot encoding for categorical columns')
print('   - Feature engineering: income-to-loan ratio, age bands')
print('   - Stratified split to preserve class ratio')

print('\n6. Model Strategy:')
print('   - ROC-AUC is the primary metric (not accuracy, due to imbalance)')
print('   - Recall for default class is critical (false negatives are costly)')
print('   - Class weighting is necessary for imbalanced training')
print('   - Threshold tuning will balance precision vs recall')

print('=' * 60)